# Certificate-Gated Defense Experiment
**Run the full attack/defense grid on Colab Pro A100 GPU**

This notebook:
1. Uploads the project code
2. Installs dependencies + loads LLM on GPU (model auto-selected by speed preset)
3. Runs 9 defenses × 6 attack strategies × seeds × tasks with a real LLM
4. Computes metrics, generates all 25 figures, and runs full analysis suite

Includes: paper figures, attack trace, attack optimization, adaptive attacks, formal attack optimization (proposal-aligned objective), L_bad/ΔL_bad correlation, planner-executor comparison, defense-in-depth analysis

Runtime depends on speed preset:
- **fast** (~5 min): 1.5B model, 3 seeds, 20/cell = ~3,240 episodes
- **balanced** (~20 min): 3B model, 3 seeds, 30/cell = ~4,860 episodes
- **full** (~120 min): 14B model, 5 seeds, 50/cell = ~13,500 episodes

## 0. Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name}")
    print(f"VRAM: {vram:.1f} GB")
    if vram < 30:
        print("⚠️  You have a smaller GPU. Go to Runtime > Change runtime type > A100 GPU")
        print("   Or change model_name below to 'Qwen/Qwen2.5-7B-Instruct' for T4/V100")
    else:
        print("✓ A100 detected — running Qwen2.5-14B-Instruct (best quality)")

## 1. Upload project
Upload your `cert-agent-exp` folder as a zip, or clone from GitHub.

In [ ]:
import os

# ── Pick ONE upload method ─────────────────────────────────────
# Option A: Upload zip (easiest — zip cert-agent-exp/ on your machine, then upload here)
from google.colab import files
uploaded = files.upload()  # upload cert-agent-exp.zip
!unzip -qo cert-agent-exp.zip

# Option B: Mount Google Drive (uncomment these, comment out Option A)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/cert-agent-exp /content/cert-agent-exp 2>/dev/null || echo "Put cert-agent-exp in your Google Drive root"

# Option C: Clone from GitHub (uncomment and edit)
# !git clone https://github.com/YOUR_USER/cert-agent-exp.git
# ───────────────────────────────────────────────────────────────

os.chdir('/content/cert-agent-exp')
!ls

## 2. Install dependencies

In [ ]:
!pip install -q pydantic pyyaml tqdm rich orjson regex scikit-learn matplotlib numpy \
    sentence-transformers faiss-cpu datasets torch transformers accelerate

In [ ]:
import os, sys, glob

# Find project root (wherever src/cert_agent_exp exists)
for candidate in [os.getcwd(), '/content/cert-agent-exp', '/content/cert_agent_exp', '/content']:
    src = os.path.join(candidate, 'src')
    if os.path.isdir(os.path.join(src, 'cert_agent_exp')):
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Cannot find cert_agent_exp/src — check your upload path")

os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
os.environ['PYTHONPATH'] = os.path.join(PROJECT_ROOT, 'src')

print(f"Project root: {PROJECT_ROOT}")
print(f"PYTHONPATH:   {os.environ['PYTHONPATH']}")

from cert_agent_exp.models import generate
print("Imports OK")

## 3. Test the LLM on GPU

In [ ]:
import torch
vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0

# ── SPEED PRESET ──────────────────────────────────────────────
# "fast"    → 1.5B model, 3 seeds, 20/cell  (~3-5 min)
# "balanced"→ 3B model,   3 seeds, 30/cell  (~10-15 min)
# "full"    → 14B model,  5 seeds, 50/cell  (~60-90 min)
SPEED = "fast"  # <── change this to "balanced" or "full" if you want higher quality
# ──────────────────────────────────────────────────────────────

_presets = {
    "fast":     ("Qwen/Qwen2.5-1.5B-Instruct", 3, 20),
    "balanced": ("Qwen/Qwen2.5-3B-Instruct",   3, 30),
    "full":     ("Qwen/Qwen2.5-14B-Instruct",  5, 50),
}
MODEL, N_SEEDS, MAX_PER_CELL = _presets[SPEED]

print(f"Speed preset: {SPEED}")
print(f"Model: {MODEL}  |  Seeds: {N_SEEDS}  |  Episodes/cell: {MAX_PER_CELL}")
print(f"Total episodes: {9 * N_SEEDS * MAX_PER_CELL}")
print(f"Loading model...")

result = generate(
    'What is 2+2? Answer with just the number.',
    mode='hf',
    model_name=MODEL,
    temperature=0.0,
)
print(f"Model response: {result}")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## 4. Configure the grid

In [ ]:
import yaml

config = {
    'data_dir': 'data',
    'runs_dir': 'runs',
    'retrieval_mode': 'faiss',
    'use_injected_corpus': True,
    'agent': {
        'type': 'retrieval_echo',
        'max_steps': 12,
    },
    'models': {
        'mode': 'hf',
        'model_name': MODEL,  # auto-selected in previous cell
        'temperature': 0.2,
        'seed': 0,
    },
    'grid': {
        'datasets': ['hotpotqa'],
        'defenses': [
            'none', 'quote_only', 'provenance_tags', 'allowlist',
            'quote+prov+allowlist', 'certificate_gating',
            'taskshield', 'llm_judge', 'intentguard',
        ],
        'channels': ['retrieval'],
        'strategies': [
            'non_adaptive', 'goal_laundering', 'evidence_laundering',
            'policy_mimicry', 'subtle_redirect', 'footnote_injection',
        ],
        'B_tokens': [50, 150, 300],
        'K_sources': [1, 2],
        'seeds': list(range(N_SEEDS)),
        'max_per_cell': MAX_PER_CELL,
    },
}

os.makedirs('configs', exist_ok=True)
with open('configs/grid_colab.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

n_def = len(config['grid']['defenses'])
n_strat = len(config['grid']['strategies'])
n_episodes = n_def * n_strat * N_SEEDS * MAX_PER_CELL
print(f"Grid: {n_episodes} total episodes ({n_def} defenses × {n_strat} strategies × {N_SEEDS} seeds × {MAX_PER_CELL}/cell)")
print(f"Model: {MODEL} on GPU")
print(f"Estimated time: ~{n_episodes * 1.5 / 60:.0f}-{n_episodes * 3 / 60:.0f} min on A100")

## 4b. Build data pipeline (download → corpus → tasks → inject)
This creates the `data/` directory with HotpotQA tasks and injected corpus. Only needs to run once.

In [ ]:
%%time
import os, sys, json

os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

for d in ['data/raw', 'data/corpus', 'data/tasks', 'data/indexes', 'data/corpus_injected']:
    os.makedirs(d, exist_ok=True)

# Step 1: Download HotpotQA directly (no package imports needed)
print("=" * 60)
print("STEP 1: Downloading HotpotQA dataset...")
print("=" * 60)

from datasets import load_dataset
dset = load_dataset("hotpot_qa", "distractor", split="train")
dset = dset.select(range(min(2000, len(dset))))

with open("data/raw/hotpotqa-train.jsonl", "w") as f:
    for row in dset:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"[ok] {len(dset)} examples -> data/raw/hotpotqa-train.jsonl")

# Steps 2-4: Build corpus, tasks, inject (via scripts)
print("\n" + "=" * 60)
print("STEP 2: Building corpus + FAISS index...")
print("=" * 60)
!cd {PROJECT_ROOT} && PYTHONPATH=src python scripts/01_build_corpus.py --config configs/datasets.yaml

print("\n" + "=" * 60)
print("STEP 3: Generating task specs...")
print("=" * 60)
!cd {PROJECT_ROOT} && PYTHONPATH=src python scripts/02_generate_tasks.py --config configs/datasets.yaml --max_tasks 100

print("\n" + "=" * 60)
print("STEP 4: Injecting attack payloads into corpus...")
print("=" * 60)
!cd {PROJECT_ROOT} && PYTHONPATH=src python scripts/03_inject_corpus.py --config configs/attacks.yaml

# Verify
print("\n" + "=" * 60)
print("VERIFICATION:")
print("=" * 60)
all_ok = True
for f in ['data/raw/hotpotqa-train.jsonl', 'data/corpus/chunks.jsonl',
          'data/tasks/hotpotqa_tasks.jsonl',
          'data/corpus_injected/injection_manifest.json',
          'data/indexes/faiss_flatip.index']:
    path = os.path.join(PROJECT_ROOT, f)
    exists = os.path.exists(path)
    tag = "OK" if exists else "MISSING"
    all_ok = all_ok and exists
    size = f"{os.path.getsize(path)/1e6:.1f} MB" if exists else ""
    print(f"  [{tag}] {f}  {size}")

if all_ok:
    print("\nAll data files ready!")

## 5. Run the full grid

In [ ]:
import time, yaml, os

cfg_path = os.path.join(PROJECT_ROOT, 'configs', 'grid_colab.yaml')
os.makedirs(os.path.dirname(cfg_path), exist_ok=True)

# Always (re)write config from current preset variables
config = {
    'data_dir': 'data',
    'runs_dir': 'runs',
    'retrieval_mode': 'faiss',
    'use_injected_corpus': True,
    'agent': {'type': 'retrieval_echo', 'max_steps': 12},
    'models': {'mode': 'hf', 'model_name': MODEL, 'temperature': 0.2, 'seed': 0},
    'grid': {
        'datasets': ['hotpotqa'],
        'defenses': [
            'none', 'quote_only', 'provenance_tags', 'allowlist',
            'quote+prov+allowlist', 'certificate_gating',
            'taskshield', 'llm_judge', 'intentguard',
        ],
        'channels': ['retrieval'],
        'strategies': [
            'non_adaptive', 'goal_laundering', 'evidence_laundering',
            'policy_mimicry', 'subtle_redirect', 'footnote_injection',
        ],
        'B_tokens': [50, 150, 300],
        'K_sources': [1, 2],
        'seeds': list(range(N_SEEDS)),
        'max_per_cell': MAX_PER_CELL,
    },
}
with open(cfg_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

total = 9 * 6 * N_SEEDS * MAX_PER_CELL
print(f"Running grid: {total} episodes  |  mode=hf  |  model={MODEL}")
start = time.time()

!cd {PROJECT_ROOT} && PYTHONPATH=src python scripts/05_run_grid.py --config {cfg_path}

elapsed = time.time() - start
print(f"\n=== Grid completed in {elapsed/60:.1f} minutes ===")

## 6. Compute metrics

In [ ]:
!cd {PROJECT_ROOT} && PYTHONPATH=src python scripts/06_compute_metrics.py --config configs/grid_colab.yaml

## 7. Generate all figures

In [ ]:
# Performance figures (ASR with CI, exposure, security, heatmap)
!cd {PROJECT_ROOT} && PYTHONPATH=src python scripts/07b_plot_performance.py --config configs/grid_colab.yaml

# Paper figures (tradeoff, tau sensitivity, bar chart, architecture)
!cd {PROJECT_ROOT} && PYTHONPATH=src python scripts/11_generate_paper_figures.py

# Attack trace figure (input → injection → output → defense)
!cd {PROJECT_ROOT} && PYTHONPATH=src python scripts/12_attack_trace_figure.py

In [ ]:
from IPython.display import Image, display
import glob

for fig in sorted(glob.glob('runs/figures/*.png')):
    print(f"\n--- {os.path.basename(fig)} ---")
    display(Image(filename=fig, width=700))

## 8. Run internal proof package

In [ ]:
!cd {PROJECT_ROOT} && PYTHONPATH=src python scripts/09_proof_package.py --config configs/grid_colab.yaml

In [ ]:
for fig in sorted(glob.glob('runs/proof/*.png')):
    print(f"\n--- {os.path.basename(fig)} ---")
    display(Image(filename=fig, width=700))

## 9. Live demo: watch attack & defense

In [ ]:
!cd {PROJECT_ROOT} && PYTHONPATH=src python scripts/10_live_demo.py --config configs/grid_colab.yaml --defense none --fast

In [ ]:
!cd {PROJECT_ROOT} && PYTHONPATH=src python scripts/10_live_demo.py --config configs/grid_colab.yaml --defense certificate_gating --fast

In [ ]:
!cd {PROJECT_ROOT} && PYTHONPATH=src python scripts/10_live_demo.py --config configs/grid_colab.yaml --compare-all --fast

## 10. Attack Optimization
Search over 288 payload × strategy × budget combinations to find the best attack against taint detection.

In [ ]:
!cd {PROJECT_ROOT} && PYTHONPATH=src python scripts/13_attack_optimization.py

## 11. Adaptive Attacks
Evaluate goal laundering, evidence laundering, and policy mimicry against certificate validation + taint detection.

In [ ]:
!cd {PROJECT_ROOT} && PYTHONPATH=src python scripts/14_adaptive_attack_analysis.py

## 12. Generate attack figures
Strategy scores, payload rankings, adaptive detection, defense-in-depth summary.

In [ ]:
!cd {PROJECT_ROOT} && PYTHONPATH=src python scripts/15_attack_figures.py

# Display all attack figures
from IPython.display import Image, display
import glob

for fig in sorted(glob.glob('runs/figures/attack_*.png')) + \
          sorted(glob.glob('runs/figures/adaptive_*.png')) + \
          sorted(glob.glob('runs/figures/defense_in_depth.png')):
    print(f"\n--- {os.path.basename(fig)} ---")
    display(Image(filename=fig, width=700))

## 12b. Extended results + formal attack optimization + L_bad correlation + planner-executor

In [ ]:
import os
os.chdir(PROJECT_ROOT)

# Extended results (ablation, budget experiment, heatmap, cert verification stats)
!PYTHONPATH=src python scripts/16_extended_results.py

# Formal attack optimization: max_δ E[1{a∈B}] - λ·ℓ_task
!PYTHONPATH=src python scripts/17_formal_attack_optimization.py

# L_bad / ΔL_bad correlation analysis
!PYTHONPATH=src python scripts/18_lbad_correlation.py

# Planner-executor secondary experiment
!PYTHONPATH=src python scripts/19_planner_executor_experiment.py --skip-run

In [ ]:
# Display ALL figures (25 total)
from IPython.display import Image, display
import glob, os

os.chdir(PROJECT_ROOT)
figs = sorted(glob.glob('runs/figures/*.png'))
print(f"Total figures: {len(figs)}\n")
for fig in figs:
    print(f"\n--- {os.path.basename(fig)} ---")
    display(Image(filename=fig, width=700))

## 13. Sample LLM outputs (inspect what the model actually said)

In [ ]:
import json

logs = []
with open('runs/logs/grid_run.jsonl') as f:
    for line in f:
        logs.append(json.loads(line))

print(f"Total episodes: {len(logs)}\n")

# Show one example per defense
seen = set()
for L in logs:
    d = L.get('defense', '')
    if d in seen:
        continue
    seen.add(d)
    pa = L.get('parsed_action', {})
    content = (pa.get('content') or '')[:300]
    outcome = L.get('defense_trace', {}).get('final_outcome', '?')
    success = L.get('success', '?')
    print(f"=== {d} === outcome={outcome}  task_success={success}")
    print(f"  LLM output: {content}...")
    print()

## 14. Download results

In [ ]:
!zip -r results.zip runs/logs/ runs/metrics/ runs/figures/ runs/proof/ \
    runs/attack_optimization.json runs/adaptive_attack_results.json \
    runs/formal_attack_optimization.json runs/lbad_correlation.json \
    runs/extended_results.json \
    REPORT.md docs/STATUS.md

from google.colab import files
files.download('results.zip')